## Import knižníc

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from catboost import CatBoostClassifier, Pool
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder 
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, RandomizedSearchCV
from imblearn.over_sampling import SMOTENC, SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    make_scorer,
    recall_score,
    confusion_matrix,
    classification_report
)
from sklearn.inspection import permutation_importance

from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

import shap
from matplotlib.colors import ListedColormap


## Pochopenie a príprava dát

### Načítanie všetkých vĺn

In [ ]:
w1 = pd.DataFrame(pd.read_excel('data/VEGA_dáta z 13-11-2024/spracovane_Pavol-Almasi/1. vlna všetko 28-11-2024.xlsx'))
w2 = pd.DataFrame(pd.read_excel('data/VEGA_dáta z 13-11-2024/spracovane_Pavol-Almasi/2. vlna všetko 28-11-2024.xlsx'))
w3 = pd.DataFrame(pd.read_excel('data/VEGA_dáta z 13-11-2024/spracovane_Pavol-Almasi/3. vlna všetko 28-11-2024.xlsx'))
w4 = pd.DataFrame(pd.read_excel('data/VEGA_dáta z 13-11-2024/spracovane_Pavol-Almasi/4. vlna všetko 28-11-2024.xlsx'))

### Spojenie vĺn do jednej množiny
Pred sojením všetkých vĺn do jednej množiny sa kontroluje ich konzistencia, aby sa zebezpečilo, že všetky súory majú identické stĺpce.

In [ ]:
waves = [w1, w2, w3, w4]

columns = [set(df.columns) for df in waves]

base_cols = columns[0]
all_match = all(cols == base_cols for cols in columns)

if all_match:
    print('Correct columns.')

    for i, df in enumerate(waves, start=1):
        df['wave_id'] = i

    df_all = pd.concat(waves, axis=0, ignore_index=True)
    print('df_all shape: ', df_all.shape)

else:
    print('Incorrect columns.')
    for i, cols in enumerate(columns, start=1):
        missing = base_cols - cols
        extra = cols - base_cols
        print(f'Vlna {i} \nMissing: {missing}\nExtra: {extra}')

### Definícia cieľového atribútu LOS
Na zákalde dátumov prijatia a prepustenia sa vypčíta celková dĺžka hospitalizácie v dňoch, ktorá sa potom diskretizuje na 3 intervaly.

In [ ]:
df_all['Dátum príjmu'] = pd.to_datetime(df_all['Dátum príjmu'], dayfirst=True, errors='coerce')
df_all['Dátum prepustenia'] = pd.to_datetime(df_all['Dátum prepustenia'], dayfirst=True, errors='coerce')
df_all['dlzka_hospitalizacie'] = (df_all['Dátum prepustenia'] -  df_all['Dátum príjmu']).dt.days

intervals = pd.IntervalIndex.from_tuples([(0, 8), (8, 15), (15, float('inf'))], closed='left')

bins = [0, 8, 15, float('inf')]
labels = ['0-7', '8-14', '15+']

df_all['LOS'] = pd.cut(
    df_all['dlzka_hospitalizacie'],
    bins=bins,
    labels=labels,
    right=False
)

#### Barplot pre distribúciu cieľového atribútu

In [ ]:
counts = df_all['LOS'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
sns.set_style('whitegrid') 

ax = sns.barplot(x=counts.index.astype(str), y=counts.values, palette='Blues_d', width=0.6)

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), 
                   (p.get_x() + p.get_width() / 2., p.get_height()), 
                   ha = 'center', va = 'center', 
                   xytext = (0, 15), 
                   textcoords = 'offset points',
                   fontsize=24, fontweight='bold') 

plt.title('Distribúcia tried dĺžky hospitalizácie (LOS)', fontsize=22, pad=20)
plt.xlabel('Trieda LOS', fontsize=22, labelpad=10)
plt.ylabel('Počet pacientov', fontsize=22, labelpad=10)

plt.xticks(fontsize=24)
plt.yticks(fontsize=18)
plt.ylim(0, max(counts.values) * 1.2)

sns.despine()
plt.tight_layout()

# Uloženie obrázka
# plt.savefig('graf_los.png', dpi=300, bbox_inches='tight')

plt.show()

### Extrakcia ďalších atribútov
Extrahovanie atribútov pomocov REGEX

In [ ]:
# teplota a tlak

def extract_tk(text):
    if pd.isna(text):
        return np.nan
    match = re.search(r'TK[:\s]*([0-9]{2,3})[\/\-]([0-9]{2,3})', text)
    if match:
        syst, diast = match.groups()
        return f'{syst}/{diast}'
    return np.nan

def extract_tt(text):
    if pd.isna(text):
        return np.nan
    match = re.search(r'TT[:\s]*([0-9]{2,3}[.,]?\d*)\s*°C', text)
    if match:
        temp = match.group(1).replace(',', '.')
        return float(temp)
    return np.nan

df_all = df_all.copy()

df_all['tlak'] = df_all['Objektívny nález'].apply(extract_tk)
df_all['teplota'] = df_all['Objektívny nález'].apply(extract_tt)

df_all[['TK_syst', 'TK_diast']] = df_all['tlak'].str.split('/', expand=True)

df_all['TK_syst'] = pd.to_numeric(df_all['TK_syst'], errors='coerce')
df_all['TK_diast'] = pd.to_numeric(df_all['TK_diast'], errors='coerce')

In [ ]:
# CT severity score

CT_cols = ['Epikríza', 'SVLZ správy', 'Osobná anamnéza']

def extract_ct_score_from_text(text):
    if pd.isna(text):
        return None
    match = re.search(r'CT severity score\s*[:]? (\d+)[\/| ]', text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

def extract_ct_score(row):
    for col in CT_cols:
        score = extract_ct_score_from_text(row[col])
        if score is not None:
            return score
    return np.nan

df_all['CT_severity_score'] = df_all.apply(extract_ct_score, axis=1)

### Redukcia dát

#### Vynechanie stĺpcov, ktoré majú aspoň 60% chýbajúcich hodnôt

In [ ]:
missing_counts = df_all.isnull().sum()
missing_percent = (missing_counts / len(df_all)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_percent': missing_percent
}).sort_values(by='missing_percent', ascending=False)

high_missing = missing_df[missing_df['missing_percent'] >= 60]

print('Atribúty s viac ako 60 % chýbajúcich hodnôt:')
print(high_missing)

plt.figure(figsize=(12, 6))
sns.barplot(
    y=high_missing.index,
    x=high_missing['missing_percent'],
    palette='rocket',
    hue=high_missing.index,
    legend=False
)
plt.title('Atribúty s viac ako 60 % chýbajúcich hodnôt')
plt.xlabel('Percento chýbajúcich hodnôt (%)')
plt.ylabel('Atribút')
plt.tight_layout()
plt.show()


df_all = df_all.loc[:, df_all.isnull().mean() <= 0.6]

#### Vynechanie záznamov, ktorých laboratórne atribúty majú 100% c hýhbajúcich hodnôt

In [ ]:
lab_cols = df_all.select_dtypes(include=['float64']).columns[:-3].difference(['Závažnosť priebehu ochorenia', 'SatO2 %']).tolist()

df_missing = df_all.copy()

df_missing['missing_pct'] = df_missing[lab_cols].isna().sum(axis=1) / len(lab_cols) * 100

df_missing.groupby(pd.cut(df_missing['missing_pct'], [50,60,70,80,90,100, float('inf')], right=False)).size()

df_all = df_all.loc[
    df_missing['missing_pct'] < 100
].reset_index(drop=True)

### Analýza LOS na základe mortality pacientov

Definícia premennej mortalita pomocou atribútu 'Závažnosť priebehu ochorenia'

In [ ]:
df_all['mortalita'] = np.where(
    df_all['Závažnosť priebehu ochorenia'].isna(),
    np.nan,
    np.where(df_all['Závažnosť priebehu ochorenia'] == 3, 1, 0)
)

#### Analýza

In [ ]:
df_all.groupby('mortalita')['dlzka_hospitalizacie'].describe()

In [ ]:
plt.figure(figsize=(8, 10))
sns.set_style('whitegrid')

ax = sns.boxplot(
    x='mortalita', 
    y='dlzka_hospitalizacie', 
    data=df_all, 
    palette='Paired', 
    width=0.7,
    fliersize=10,
    linewidth=3   
)

plt.title('Dĺžka hospitalizácie podľa mortality', fontsize=24, pad=25, fontweight='bold')
plt.xlabel('Mortalita (0 = prežil, 1 = zomrel)', fontsize=24, labelpad=15)
plt.ylabel('Dĺžka hospitalizácie', fontsize=20, labelpad=15)
plt.xticks(fontsize=24)
plt.yticks(fontsize=20)
sns.despine()
plt.tight_layout()

# Uloženie obrázka
# plt.savefig('boxplot_mortalita_los.png', dpi=300, bbox_inches='tight')

plt.show()


### Vynechanie nepotrebných atribútov pre ďalšie spracovanie 

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_seq_items', None)

result = (
    df_all.dtypes.reset_index()
      .rename(columns={'index': 'column', 0: 'dtype'})
      .groupby('dtype', sort=False)['column']
      .apply(list)
)

print(result.to_string())

id_cols = ['Meno', 'wave_id']
for_later_prep = ['Epikríza', 'Objektívny nález', 'Osobná anamnéza', 'DRG výkony', 'SVLZ správy']
redundant_cols = ['Poradie','Kód prepustenia','Dátum príjmu', 'Dátum prepustenia', 'dlzka_hospitalizacie', 'mortalita',
                  'Liečba','HLN Dg.', 'Diagnózy', 'Dôvod hospitalizácie', 'Lieková anamnéza', 'Návyková anamnéza',
                  'Epidemiologická anamnéza', 'tlak', 'Terajšie ochorenie', 'Závažnosť priebehu ochorenia']
col_to_drop = redundant_cols + for_later_prep
df_all_cleaned = df_all.drop(columns=col_to_drop)

### Ošetrenie tehcnicky chybných hodnôt

In [ ]:
# Povolený rozsah hodnôt 
ranges = {
    's-bil-t': (1.0, 600.0),
    's-ast': (0.1, 150.0),
    's-alt': (0.1, 150.0),
    's-gmt': (0.1, 150.0),
    's-alp': (0.1, 50.0),
    's-cb': (10.0, 150.0),
    's-na': (100.0, 200.0),
    's-k': (1.0, 10.0),
    's-cl': (70.0, 150.0),
    's-crp': (0.0, 2000.0),
    's-alb': (10.0, 100.0),
    's-gluk': (0.1, 50.0),
    's-urea': (0.5, 50.0),
    's-kreat': (10.0, 2000.0),
    's-km': (60.0, 1000.0),
    's-ck': (0.1, 50.0),
    's-ck-mb': (0.0, 20.0),
    'P-Laktát': (0.0, 10.0),
    's-fer': (1.0, 10000.0),
    's-il6': (0.1, 10000.0),
    'hgb': (0.0, 20.0),
    'wbc': (0.1, 60.0),
    'plt': (0.0, 3000.0),
    'neu abs': (0.0, 50.0),
    'eo abs': (0.0, 10.0),
    'ly abs': (0.0, 15.0),
    'pt (inr)': (0.2, 8.0),
    'aptt-r': (0.2, 8.0),
    'fib': (0.0, 10.0),
    'cd3+': (0.0, 5.0),
    'cd4+': (0.0, 5.0),
    'cd8+': (0.0, 5.0),
    'cd4+/cd8+': (0.0, 5.0),
    'pdw': (0.0, 50.0),
    'cd19+': (0.0, 5.0),
    'nk': (0.0, 5.0),
    'ne/ly': (0.0, 50.0),
    'D-dimér': (0.0, 10.0),
    'SatO2 %': (30.0, 100.0),
    's-pbnp': (0.0, 50000.0),
    's-vitd': (0.0, 1000.0),
    'teplota': (33.0, 41.0),
    'TK_syst': (60.0, 250.0),
    'TK_diast': (30.0, 170.0)

}

# zoradenie prefixov podľa dĺžky
sorted_prefixes = sorted(ranges.keys(), key=lambda x: len(x), reverse=True)

def check_out_of_range_values(input_df):
    error_values_summary = []

    for col in input_df.columns:
        col_lower = col.lower().strip()
        matched_prefix = None

        # hľadanie najdlhšieho zodpovedajúceho prefixu
        for prefix in sorted_prefixes:
            if col_lower.startswith(prefix.lower()):
                matched_prefix = prefix
                break

        if matched_prefix:
            low, high = ranges[matched_prefix]

            # logika pre min/max stĺpce
            if col_lower.endswith('min'):
                mask = input_df[col] < low
            elif col_lower.endswith('max'):
                mask = input_df[col] > high
            else:
                mask = (input_df[col] < low) | (input_df[col] > high)

            error_values_count = mask.sum()
            if error_values_count > 0:
                error_values_summary.append({
                    'column': col,
                    'error_values_count': error_values_count,
                    'total_values': input_df[col].shape[0],
                    'error_%': float(error_values_count / input_df[col].shape[0]),
                    'count_nan_%': input_df[col].isna().mean(),
                    'sum_nan_%': (float(error_values_count/input_df[col].shape[0]) +
                                 input_df[col].isna().mean())*100,
                })

    if error_values_summary:
        summary_df = pd.DataFrame(error_values_summary)
        display(summary_df.sort_values(by='error_values_count', ascending=False))
    else:
        print('Žiadne hodnoty mimo definovaných intervalov.')

In [ ]:
check_out_of_range_values(df_all_cleaned)

#### Oprava teploty a tlaku 

In [ ]:
df_all[['Meno','teplota','wave_id']][
    (df_all['teplota'] < 33) | (df_all['teplota'] > 41)
]

df_all.loc[df_all['teplota'] > 45, 'teplota'] = df_all.loc[df_all['teplota'] > 45, 'teplota'] / 10

In [ ]:
df_all[['wave_id','Meno','TK_syst', 'TK_diast']][
    (df_all['TK_syst'])<(df_all['TK_diast'])
]

mask = df_all['TK_syst'] < df_all['TK_diast']

# oprava TK_syst: ak má 2 cifry, pridá sa 0 na koniec
df_all.loc[mask, 'TK_syst'] = df_all.loc[mask, 'TK_syst'].apply(
    lambda x: int(str(int(x)) + '0') if 10 <= x <= 99 else x
)

# oprava TK_diast: ak má 3 cifry, nechajú sa len prvé 2
df_all.loc[mask, 'TK_diast'] = df_all.loc[mask, 'TK_diast'].apply(
    lambda x: int(str(int(x))[:2]) if x >= 100 else x
)

#### Oprava ostatných atribútov nahradením s NaN

In [ ]:
for col in df_all_cleaned.columns:
    col_lower = col.lower().strip()
    matched_prefix = None

    for prefix in sorted_prefixes:
        if col_lower.startswith(prefix.lower()):
            matched_prefix = prefix
            break

    if matched_prefix:
        low, high = ranges[matched_prefix]

        if col_lower.endswith('min'):
            mask = df_all_cleaned[col] < low
        elif col_lower.endswith('max'):
            mask = df_all_cleaned[col] > high
        else:
            mask = (df_all_cleaned[col] < low) | (df_all_cleaned[col] > high)

        # nahradenie NaN
        df_all_cleaned.loc[mask, col] = np.nan

check_out_of_range_values(df_all_cleaned)

### Konverzia dátových typov pre zníženie pamäťovej náročnosti a optimalizácie pre modelovanie.


In [ ]:
def dtype_check(df):
    dtypes_grouped = (
        df.dtypes.astype(str)
            .groupby(df.dtypes.astype(str))
            .groups
    )

    for dtype, cols in dtypes_grouped.items():
        print(f'\n{dtype}:')
        print(list(cols))

dtype_check(df_all_cleaned)

In [ ]:
def int_to_float(df, cols):

    for col in cols:
        if col in df.columns:          
            df[col] = df[col].astype('float64')
        else:
            print(f'Warning: Column "{col}" not found in DataFrame.')
    return df

df_all_cleaned = int_to_float(df_all_cleaned, ['Vek', 'Počet dávok'])

def obj_to_cat(df, cols):

    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
        else:
            print(f'Warning: Column "{col}" not found in DataFrame.')
    return df

df_all_cleaned = obj_to_cat(df_all_cleaned, ['Pohlavie', 'Kód príjmu', 'wave_id'])

### Rozdelenie datasetu

In [ ]:
all_but_last = df_all_cleaned[df_all_cleaned['wave_id'] != 4]
first = df_all_cleaned[df_all_cleaned['wave_id'] == 1]
second = df_all_cleaned[df_all_cleaned['wave_id'] == 2]
third = df_all_cleaned[df_all_cleaned['wave_id'] == 3]
last = df_all_cleaned[df_all_cleaned['wave_id'] == 4]

In [ ]:
datasety = (all_but_last, first, second, third, last)
for dt in datasety:
    print(len(dt))

### Benchmark model

In [ ]:
def train_benchmark_model(train_df, features=None,
                          weights=None, smote=False, ros=False,
                          fs=False, return_metrics=False):

    id_cols = ['Meno', 'wave_id']

    if features:
        X_train = train_df[features]
        y_train = train_df['LOS']
    else:
        X_train = train_df.drop(columns= id_cols + ['LOS'])
        y_train = train_df['LOS']

    cat_features = X_train.select_dtypes(include=['category', 'bool']).columns.tolist()
    cat_features_indices = [X_train.columns.get_loc(c) for c in cat_features]

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_scores = []
    fs_results = pd.DataFrame(index=X_train.columns)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    
        print(f'\n===== Fold {fold+1} =====')
        
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # -------------------------------------------- ROS -------------------------------------------
        if ros:
            print('--- STAV PRED ROS ---')
            print(y_tr.value_counts().sort_index())
            print('-' * 30)
            ros = RandomOverSampler(random_state=42)
            X_tr, y_tr = ros.fit_resample(X_tr, y_tr)

            print('--- STAV PO ROS ---')
            print(y_tr.value_counts().sort_index())
            print('-' * 30)

        # -------------------------------------------- SMOTE -------------------------------------------
        if smote:
            print('--- STAV PRED SMOTE ---')
            print(y_tr.value_counts().sort_index())
            print('-' * 30)

            smote_features = X_tr.select_dtypes(include=['category', 'bool']).columns.tolist()
            smote_features_indices = [X_tr.columns.get_loc(c) for c in smote_features]

            if len(smote_features_indices) > 0: 
                sampler = SMOTENC(categorical_features=smote_features_indices, random_state=42)
                
            else: 
                sampler = SMOTE(random_state=42)
            
            X_tr, y_tr = sampler.fit_resample(X_tr, y_tr)
                
            print('--- STAV PO SMOTE ---')
            print(y_tr.value_counts().sort_index())
            print('-' * 30)

        # ----------------------------------------------------------------------------------------------

        model = CatBoostClassifier(
            eval_metric='TotalF1:average=Macro',
            loss_function='MultiClass',
            auto_class_weights=weights,
            early_stopping_rounds=100,
            random_seed=42,
            thread_count=-1,
            verbose=100
        )

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_features_indices,
            eval_set=(X_val, y_val),
            use_best_model=True
        )

        # -------------------------------------- Výber atribútov ----------------------------------------
        if fs:
            result = permutation_importance(model,
                                            X_val,
                                            y_val,
                                            scoring='f1_macro',
                                            n_jobs=-1,
                                            random_state=42)

            selected_features = X_tr.columns[result.importances_mean > 0].tolist()

            column_name = f'Fold_{fold}'
            fs_results[column_name] = 0 
            fs_results.loc[selected_features, column_name] = 1

        # ------------------------------------- Vyhodnotenie foldu ------------------------------------------
        y_pred = model.predict(X_val)
        y_pred_proba = model.predict_proba(X_val)

        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro')
        rec = recall_score(y_val, y_pred, average='macro')
        auc = roc_auc_score(y_val, y_pred_proba, multi_class='ovr', average='macro')
        fold_scores.append({'fold': fold+1, 'accuracy': acc, 'macro_f1': f1, 'macro_recall': rec, 'auc': auc})

        print(f'Macro F1: {f1:.4f}, Macro Recall: {rec:.4f}, AUC(OvR): {auc:.4f}, Accuracy: {acc:.4f}')

    # -------------------------------------------- Vyhodnotenie CV ------------------------------------------
    df_scores = pd.DataFrame(fold_scores)
    mean_metrics = df_scores.mean()
    std_metrics = df_scores.std()

    print('\n---- K-Fold CV Summary ----')
    print(df_scores)
    print('\nMean metrics:')
    print(mean_metrics)
    print('\nStd metrics:')
    print(std_metrics)

    if fs:
        return fs_results
    
    if return_metrics:
        results_summary = {
            'f1_mean': mean_metrics['macro_f1'],
            'f1_std': std_metrics['macro_f1'],
            'recall_mean': mean_metrics['macro_recall'],
            'recall_std': std_metrics['macro_recall'],
            'auc_mean': mean_metrics['auc'],
            'auc_std': std_metrics['auc'],
            'accuracy_mean': mean_metrics['accuracy'],
            'accuracy_std': std_metrics['accuracy']
        }
        return results_summary

### Ošetrenie chýbajúcich hodnôt

Analýza chýbajúcich hodnôt

In [ ]:
def check_null(dfs):
    for df in dfs:
        missing = (
            df.isnull()
                .sum()
                .groupby(df.dtypes.astype(str))
                .sum()
        )
        print(missing)

check_null([df_all_cleaned])

#### Imputácia mediánom

In [ ]:
def median_impute(train_df, test_df):

    train = train_df.copy()
    test = test_df.copy()

    columns = train.select_dtypes(include=['float64']).columns
    
    imputer = SimpleImputer(strategy='median')

    train[columns] = imputer.fit_transform(train[columns])
    test[columns] = imputer.transform(test[columns])
    
    return train, test

In [ ]:
v1_train_imputed_median, v1_test_imputed_median = median_impute(all_but_last, last)
train_first_imputed_median, test_second_imputed_median = median_impute(first, second)
train_second_imputed_median, test_third_imputed_median = median_impute(second, third)
train_third_imputed_median, test_last_imputed_median = median_impute(third, last)

#### Imputácia s MICE

Pomocná funkcia pre vytvorenie hodnôt `min_value` a `max_value` pre `IterativeImputer`

In [ ]:
def prepare_min_max_for_imputer(numeric_cols, ranges, df):

    sorted_prefixes = sorted(ranges.keys(), key=lambda x: len(x), reverse=True)

    min_values = []
    max_values = []

    for col in numeric_cols:
        col_lower = col.lower().strip()
        matched_prefix = None

        # najdlhší prefix pre pokrytie agregovaných stĺpcov
        for prefix in sorted_prefixes:
            if col_lower.startswith(prefix.lower()):
                matched_prefix = prefix
                break

        if matched_prefix:
            min_val, max_val = ranges[matched_prefix]

        else:
            # ak prefix nie je definovaný, použijú sa empirické hranice 
            min_val = df[col].min()
            max_val = df[col].max()

        min_values.append(min_val)
        max_values.append(max_val)

    return min_values, max_values

PCA analýza

In [ ]:
imputer = SimpleImputer(strategy='median')
df_filled = imputer.fit_transform(first.select_dtypes(include=['float64']))

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_filled)

pca = PCA()
pca.fit(df_scaled)

cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

n_90 = np.argmax(cumulative_variance >= 0.90) + 1
n_95 = np.argmax(cumulative_variance >= 0.95) + 1

print(f'Počet čŕt pre 90% rozptylu: {n_90}')
print(f'Počet čŕt pre 95% rozptylu: {n_95}')

plt.rcParams.update({'font.size': 30})
plt.figure(figsize=(12, 8))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.axhline(y=0.90, color='r', linestyle='-', label='90% Threshold')
plt.axvline(x=n_90, color='r', linestyle='--')

plt.xlabel('Počet komponentov (čŕt)', fontsize=28)
plt.ylabel('Pomer vysvetleného rozptylu', fontsize=28)

plt.legend(fontsize=28)
plt.xticks(fontsize=26)
plt.yticks(fontsize=26) 

plt.grid(True)

# plt.savefig('pca_analysis_result.png', dpi=300, bbox_inches='tight')

plt.show()

MICE

In [ ]:
def mice_impute(train_df, test_df=None, n_nearest_features=55, n_estimators=40):

    numeric_cols = train_df.select_dtypes(include=['float64']).columns
    
    min_vals, max_vals = prepare_min_max_for_imputer(numeric_cols, ranges, train_df)

    imputer = IterativeImputer(
        estimator=ExtraTreesRegressor(n_estimators=n_estimators, n_jobs=-1, random_state=42),
        min_value=min_vals,
        max_value=max_vals,
        n_nearest_features=n_nearest_features,
        random_state=42,
        skip_complete=True,
    )

    train_copy = train_df.copy()
    train_copy[numeric_cols] = imputer.fit_transform(train_copy[numeric_cols])
    
    if test_df:
        test_copy = test_df.copy()
        test_copy[numeric_cols] = imputer.transform(test_copy[numeric_cols])
    
    return train_copy, test_copy

##### Testovanie `n_nearest_features`

In [ ]:
for n in [30, 55,74]:
    mice_impute(first, n_estimators=20, n_nearest_features=n)
    train_benchmark_model(first)

##### Testovanie `n_estimators`

In [ ]:
for n in [20, 50, 30, 40]:
    mice_impute(first, n_estimators=n, n_nearest_features=55)
    train_benchmark_model(first)

Imputácia s najlepším nastavením parametrov MICE

In [ ]:
# Verzia 1
v1_train_imputed, v1_test_imputed = mice_impute(
    all_but_last,
    last,
)

# Verzia 2
train_first_imputed, test_second_imputed = mice_impute(
    first,
    second,
)

train_second_imputed, test_third_imputed = mice_impute(
    second,
    third,
)

train_third_imputed, test_last_imputed = mice_impute(
    third,
    last,
)

#### Otestovanie imputácií na benchmark modeli

In [ ]:
# bez imputácie
train_benchmark_model(first)

# imputácia mediánom
train_benchmark_model(train_first_imputed_median)

# imputácia s MICE
train_benchmark_model(train_first_imputed)

### Riešenie nerovnováhy tried
Testovanie metód použitím váh, ROS a SMOTE

In [ ]:
train_benchmark_model(train_first_imputed, weights='Balanced')
train_benchmark_model(train_first_imputed, ros=True)
train_benchmark_model(train_first_imputed, smote=True)

### Výber atribútov

Funkcia na otestovanie všetkých prahov

In [ ]:
def evaluate_all_thresholds(train_df, features):
    
    threshold_results = []
    
    for t in range(0, 10 + 1):
        current_features = features[features['Selection_Rate'] >= t].index.tolist()
        n_feat = len(current_features)
        
        print(f'\nPrah {t}/{10} (Počet atribútov: {n_feat})...')

        if n_feat == 0:
            print(f'Skip prah {t}, nula vybraných atribútov.')
            continue

        metrics = train_benchmark_model(train_df, features=current_features, smote=True, return_metrics=True)
        
        metrics['threshold'] = f'{t}'
        metrics['n_features'] = n_feat
        threshold_results.append(metrics)

    summary_df = pd.DataFrame(threshold_results)
    
    cols_order = ['threshold', 'n_features', 
                  'f1_mean', 'f1_std',
                  'recall_mean', 'recall_std', 
                  'auc_mean', 'auc_std', 
                  'accuracy_mean', 'accuracy_std',
                 ]
    
    summary_df = summary_df[cols_order]
    return summary_df

Funkcia pre výpis výsledkov testovania, ktorá vracia zoznam všetkých atribútov zoradených podľa `Selection_Rate`

In [ ]:
def feature_selection(train_df):

    features = train_benchmark_model(train_df, smote=True, fs=True)
    features['Selection_Rate'] = features.sum(axis=1)

    features_sorted = features.sort_values(by='Selection_Rate', ascending=False)

    feats_summary_df = evaluate_all_thresholds(train_df, features_sorted)

    feats_summary_df.sort_values(
     by=['f1_mean', 'recall_mean', 'auc_mean', 'accuracy_mean'], 
    ascending=[False, False, False, False]
    )

    return features_sorted

#### Verzia 1

In [ ]:
v1_features = feature_selection(v1_train_imputed)

In [ ]:
v1_features_final = v1_features[
    v1_features['Selection_Rate'] >= 10
].index.tolist()

#### Verzia 2

In [ ]:
# vlna 1
v2_1_features = feature_selection(train_first_imputed)

# vlna 2
v2_2_features = feature_selection(train_second_imputed)

# vlna 3
v2_3_features = feature_selection(train_third_imputed)

In [ ]:
v2_1_features_final = v2_1_features[
    v2_1_features['Selection_Rate'] >= 10
].index.tolist()

v2_2_features_final = v2_2_features[
    v2_2_features['Selection_Rate'] >= 9
].index.tolist()

v2_3_features_final = v2_3_features[
    v2_3_features['Selection_Rate'] >= 9
].index.tolist()

## Modelovanie

Funkcie pre trénovanie a vyhodnotenie modelov CatBoost a RandomForest

In [ ]:
def train_final_model(
    train_df, features=None, depth=None, lr=None, l2=None):

    X_train = train_df[features]
    y_train = train_df['LOS']
 
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(sss.split(X_train, y_train))

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    print('--- STAV PRED SMOTE ---')
    print(y_tr.value_counts().sort_index())
    print('-' * 30)

    smote_features = X_tr.select_dtypes(include=['category', 'bool']).columns.tolist()
    smote_features_indices = [X_tr.columns.get_loc(c) for c in smote_features]

    smote_nc = SMOTENC(categorical_features=smote_features_indices, random_state=42)
    X_tr, y_tr = smote_nc.fit_resample(X_tr, y_tr)

    print('--- STAV PO SMOTE ---')
    print(y_tr.value_counts().sort_index())
    print('-' * 30)

    cat_features = X_tr.select_dtypes(include=['category']).columns.tolist()
    cat_features_indices = [X_tr.columns.get_loc(c) for c in cat_features]

    params = {}
    if depth is not None:
        params['depth'] = depth
    if lr is not None:
        params['learning_rate'] = lr
    if l2 is not None:
        params['l2_leaf_reg'] = l2

    model = CatBoostClassifier(
        **params,
        eval_metric='TotalF1:average=Macro',
        loss_function='MultiClass',
        early_stopping_rounds=100,
        random_seed=42,
        thread_count=-1,
        verbose=100
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=cat_features_indices,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    return model 

In [ ]:
def train_rf_model(train_df, features=None, estimators=1000, depth=6):

    X_tr = train_df[features]
    y_tr = train_df['LOS']

    print('--- STAV PRED SMOTE ---')
    print(y_tr.value_counts().sort_index())
    print('-' * 30)

    smote = SMOTE(random_state=42)
    X_tr, y_tr = smote.fit_resample(X_tr, y_tr)

    print('--- STAV PO SMOTE ---')
    print(y_tr.value_counts().sort_index())
    print('-' * 30)

    model = RandomForestClassifier(
        n_estimators=estimators, 
        max_depth=depth,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    model.fit(
        X_tr,
        y_tr,
    )

    return model

In [ ]:
# enkóder pre kategorické atribúty - RF model

def encode_cols(train_df, test_df, cols_to_encode=None):
    if cols_to_encode is None:
        cols_to_encode = ['Pohlavie', 'Kód príjmu']
    
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    
    X_train_cat = encoder.fit_transform(train_df[cols_to_encode])
    X_test_cat = encoder.transform(test_df[cols_to_encode])
    
    encoded_cols = encoder.get_feature_names_out(cols_to_encode)
    
    X_train_cat_df = pd.DataFrame(X_train_cat, columns=encoded_cols, index=train_df.index)
    X_test_cat_df = pd.DataFrame(X_test_cat, columns=encoded_cols, index=test_df.index)

    X_train_final = pd.concat([train_df.drop(cols_to_encode, axis=1), X_train_cat_df], axis=1)
    X_test_final = pd.concat([test_df.drop(cols_to_encode, axis=1), X_test_cat_df], axis=1)
    
    return X_train_final, X_test_final, encoder

In [ ]:
# encoding vybratých atribútov

def change_encoded(selected_features, encoder):

    encoded_cols = encoder.get_feature_names_out()
    original_encoded_cols = encoder.feature_names_in_
    
    new_features = []
    
    for f in selected_features:
        if f in original_encoded_cols:
            matched = [c for c in encoded_cols if c.startswith(f + '_')]
            new_features.extend(matched)
        else:
            new_features.append(f)
    
    return new_features

In [ ]:
def evaluate_final_model(model, test_df, features, return_metrics=False):

    X_test = test_df[features]
    y_test = test_df['LOS']

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)


    acc = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {acc:.3f}')

    f1_macro = f1_score(y_test, y_pred, average='macro')
    print(f'F1 macro: {f1_macro:.3f}')

    recall_macro = recall_score(y_test, y_pred, average='macro')
    print(f'Recall macro: {recall_macro:.3f}')

    try:
        auc_ovr = roc_auc_score(
            y_test,
            y_proba,
            multi_class='ovr',
            average='macro'
        )
        print(f'AUC (macro): {auc_ovr:.3f}')
    except ValueError as e:
        print(f'AUC not defined: {e}')
    

    print('\nClassification report:')
    print(classification_report(y_test, y_pred, digits=3))

    if return_metrics:
        return{
            'accuracy': acc,
            'macro_f1': f1_macro,
            'macro_recall': recall_macro,
            'auc_ovr': auc_ovr
        }

In [ ]:
# vyhodnotenie Verzie 2 - vážený priemer metrík

def eval_v2(v2_1_result, v2_2_result, v2_3_result):
    results_list = []

    experiments = [
        ('v2_1_result', v2_1_result, len(test_second_imputed)),
        ('v2_2_result', v2_2_result, len(test_third_imputed)),
        ('v2_3_result', v2_3_result, len(test_last_imputed)),
    ]

    for name, result_dict, n in experiments:
        df = pd.DataFrame([result_dict])
        df['experiment'] = name
        df['n_samples'] = n
        results_list.append(df)

    all_results = pd.concat(results_list, ignore_index=True)

    metrics = ['macro_f1', 'macro_recall', 'auc_ovr', 'accuracy']

    weighted_results = pd.DataFrame([{
        m: np.average(all_results[m], weights=all_results['n_samples'])
        for m in metrics
    }])

    print(weighted_results)

### CatBoost

#### Verzia 1

In [ ]:
v1_model = train_final_model(v1_train_imputed, features=v1_features_final)
evaluate_final_model(v1_model, v1_test_imputed, v1_features_final)

#### Verzia 2

In [ ]:
# vlna 1

v2_1_model = train_final_model(train_first_imputed, features=v2_1_features_final)
v2_1_result = evaluate_final_model(v2_1_model, test_second_imputed, v2_1_features_final, return_metrics=True)

In [ ]:
# vlna 2

v2_2_model = train_final_model(train_second_imputed, features=v2_2_features_final)
v2_2_result = evaluate_final_model(v2_2_model, test_third_imputed, v2_2_features_final, return_metrics=True)

In [ ]:
# vlna 3

v2_3_model = train_final_model(train_third_imputed, features=v2_3_features_final)
v2_3_result = evaluate_final_model(v2_3_model, test_last_imputed, v2_3_features_final, return_metrics=True)

In [ ]:
eval_v2(v2_1_result, v2_2_result, v2_3_result)

### RF - Overenie robustnosti metód 

#### Verzia 1

In [ ]:
v1_train_imputed_encoded, v1_test_imputed_encoded, enco= encode_cols(v1_train_imputed, v1_test_imputed)

v1_model_rf = train_rf_model(v1_train_imputed_encoded, features=v1_features_final)
evaluate_final_model(v1_model_rf, v1_test_imputed_encoded, v1_features_final)

#### Verzia 2

In [ ]:
train_first_imputed_encoded, test_second_imputed_encoded, enco_1 = encode_cols(train_first_imputed, test_second_imputed)
v2_feats_1 = change_encoded(v2_1_features_final, enco_1)

v2_1_model_rf = train_rf_model(train_first_imputed_encoded, features=v2_feats_1)
v2_1_result = evaluate_final_model(v2_1_model_rf, test_second_imputed_encoded, v2_feats_1, return_metrics=True)

In [ ]:
train_second_imputed_encoded, test_third_imputed_encoded, enco_2 = encode_cols(train_second_imputed, test_third_imputed)
v2_feats_2 = change_encoded(v2_2_features_final, enco_2)


v2_2_model_rf = train_rf_model(train_second_imputed_encoded, features=v2_feats_2)
v2_2_result = evaluate_final_model(v2_2_model_rf, test_third_imputed_encoded, v2_feats_2, return_metrics=True)

In [ ]:
train_third_imputed_encoded, test_last_imputed_encoded, enco_3 = encode_cols(train_third_imputed, test_last_imputed)
v2_feats_3 = change_encoded(v2_3_features_final, enco_3)

v2_3_model_rf = train_rf_model(train_third_imputed_encoded, features=v2_feats_3)
v2_3_result = evaluate_final_model(v2_3_model_rf, test_last_imputed_encoded, v2_feats_3, return_metrics=True)

In [ ]:
eval_v2(v2_1_result, v2_2_result, v2_3_result)

### Optimalizácia modelu CatBoost

In [ ]:
def tune_test_final_model(df_train, df_test, selected_features):

    X_train = df_train[selected_features]
    y_train = df_train['LOS']

    cat_features = X_train.select_dtypes(include=['category','bool']).columns.tolist()
    cat_features_indices = [X_train.columns.get_loc(c) for c in cat_features]


    # ---------------------------- SMOTE / SMOTENC ----------------------------------

    if len(cat_features_indices) > 0:
        sampler = SMOTENC(categorical_features=cat_features_indices, random_state=42)
    else:
        sampler = SMOTE(random_state=42)


    # -------------------------------- Pipeline ----------------------------------------
    pipeline = Pipeline([
        ('smote', sampler),
        ('model', CatBoostClassifier(
            loss_function='MultiClass',
            eval_metric='TotalF1:average=Macro',
            iterations=300, 
            random_seed=42,
            thread_count=-1,
            verbose=False
        ))
    ])

    # ------------------------------ Hyperparametere -----------------------------------
    param_dist = {
        'model__depth': [4, 6, 8],
        'model__learning_rate': [0.02, 0.05, 0.08],
        'model__l2_leaf_reg': [1, 3, 5, 7]
    }

    # --------------------------------- CV + scoring -------------------------------------
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    f1_macro = make_scorer(f1_score, average='macro')


    # ------------------------------- Randomized Search ----------------------------------
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=25,
        scoring=f1_macro,
        cv=cv,
        n_jobs=-1,
        random_state=42,
        verbose=2,
        return_train_score=True
    )

    search.fit(
        X_train,
        y_train,
        model__cat_features=cat_features_indices
    )

    # -------------------------------- Finálny model --------------------------------------

    best_params = search.best_params_

    print(f'\nNajlepšie parametre: "{best_params}" ')

    best_depth = best_params['model__depth']
    best_lr = best_params['model__learning_rate']
    best_l2 = best_params['model__l2_leaf_reg']

    final_model = train_final_model(df_train, features=selected_features, lr=best_lr, l2=best_l2, depth=best_depth)
    result = evaluate_final_model(final_model, df_test, selected_features, return_metrics=True)

    return final_model, result

#### Verzia 1

In [ ]:
v1_tuned, v1_result = tune_test_final_model(v1_train_imputed, v1_test_imputed, v1_features_final)

#### Verzia 2

In [ ]:
v2_1_tuned, v2_1_result = tune_test_final_model(train_first_imputed, test_second_imputed, v2_1_features_final)
v2_2_tuned, v2_2_result = tune_test_final_model(train_first_imputed, test_second_imputed, v2_2_features_final)
v2_3_tuned, v2_3_result = tune_test_final_model(train_third_imputed, test_last_imputed, v2_3_features_final)

In [ ]:
eval_v2(v2_1_result, v2_2_result, v2_3_result)

## Vyhodnotenie

Interpretovateľnosť modelu pomocou SHAP hodnôt

### SHAP barplot

In [ ]:
X_test = v1_test_imputed[v1_features_final]
y_test = v1_test_imputed['LOS']

explainer = shap.TreeExplainer(v1_tuned)
shap_values = explainer.shap_values(X_test)

class_names = v1_tuned.classes_

plt.rcParams.update({'font.size': 16}) 
plt.figure(figsize=(14, 7))
colors = ListedColormap(['#98C4E3', '#FFBFF3', '#B0DBA4']) 

shap.summary_plot(shap_values, X_test,class_names=class_names, plot_type='bar', max_display=10, show=False, color=colors)
plt.tight_layout()
# plt.savefig('shap_barplot.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

### SHAP beeswarm grafy pre každú triedu

In [ ]:
shap_values = explainer(X_test)

top_k = 10

for i, cname in enumerate(class_names):
    print(f'SHAP plot pre triedu {cname}')

    plt.rcParams.update({'font.size': 18, 'axes.labelsize': 18, 'xtick.labelsize': 18, 'ytick.labelsize': 18})

    shap_class = shap_values[..., i]
    importance = np.abs(shap_class.values).mean(axis=0)
    top_idx = np.argsort(importance)[-top_k:]

    shap_top = shap_class[:, top_idx]
    X_top = X_test.iloc[:, top_idx]

    fig = plt.figure(figsize=(14, 6)) 

    shap.plots.beeswarm(
        shap.Explanation(
            values=shap_top.values,
            base_values=shap_top.base_values,
            data=X_top.values,
            feature_names=X_top.columns.tolist()
        ),
        plot_size=None, 
        show=False
    )

    plt.title(f'SHAP Beeswarm Plot: {cname}', fontsize=18, pad=30, fontweight='bold')
    plt.gcf().axes[0].tick_params(labelsize=18)
    # plt.savefig(f'shap_beeswarm_{cname}.png', dpi=300, bbox_inches='tight', pad_inches=0.3)
    
    plt.close(fig)


## Uloženie/načítanie modelu

In [ ]:
# Uloženie
v1_tuned.save_model('catboost_model.cbm')

# Načítanie
loaded_model = CatBoostClassifier()
loaded_model.load_model('catboost_model.cbm')